# Convergent extension from directional (planar-polarised) line tension

A flat **2D** epithelial sheet — 256 cells, free boundary, `PlanarGeometry` — in which
junction tension is graded by the angle each junction makes with a **user-supplied
polarity vector**. Junctions lying *along* the polarity axis are the most contractile;
junctions *across* it are the least. The sheet responds by narrowing along the polarity
direction and lengthening across it: **convergent extension**.

The mechanism follows the planar-polarised-tension picture of vertex-model convergent
extension (Biophysical Journal, 2021,
[S0006349521008845](https://www.sciencedirect.com/science/article/pii/S0006349521008845)),
with two deliberate differences:

* the polarity axis is **any vector the user supplies** — not a fixed lattice axis. It is
  normalised on initialisation, so `(3, 3)` and `(1, 1)` are the same axis, and so is
  `(-1, -1)`;
* the angle that sets the tension is the **acute** angle between that polarity vector and
  the junction itself.

Everything — mesh, config, spec, bigraph, runs, movies, analysis — lives in this notebook.


## The model

### The angle

A half-edge is directed but a junction is not, so the quantity that can set a tension is
the **acute** angle between the junction and the polarity axis:

$$\theta \;=\; \arccos\!\big(|\hat d \cdot \hat p|\big) \;\in\; [0,\; \pi/2]$$

with $\hat d$ the unit edge vector ($d_x, d_y$ straight out of `edge_df`) and $\hat p$ the
normalised polarity vector. The absolute value is what makes $\theta$ acute: it removes
the sign of the edge (which of `srce`/`trgt` comes first is arbitrary) *and* the sign of
the polarity vector, so $\hat p$ and $-\hat p$ give identical tensions. $\theta = 0$ is a
junction parallel to the polarity axis, $\theta = \pi/2$ one perpendicular to it.

### The tension

From $\theta$ an **alignment** $a(\theta) \in [0, 1]$ is formed — $1$ parallel, $0$
perpendicular — and the tension interpolates between two configured extremes:

$$\Lambda(\theta) \;=\; \Lambda_{\min} \;+\; (\Lambda_{\max} - \Lambda_{\min})\, a(\theta)^{\,n}$$

so **the closer a junction lies to the polarity direction, the stronger its tension**.
Three profiles are available (`profile`), and `sharpness` ($n$) narrows or broadens the
high-tension cone without moving either extreme:

| `profile` | $a(\theta)$ | |
|---|---|---|
| `cos2` (default) | $\cos^2\theta = \tfrac12(1 + \cos 2\theta)$ | with $n = 1$ this is the classical nematic law $\Lambda = \bar\Lambda\,(1 + \alpha \cos 2\theta)$ |
| `abs_cos` | $\lvert\cos\theta\rvert$ | broader high-tension cone |
| `linear` | $1 - 2\theta/\pi$ | linear in the angle itself |

For `cos2` with $n=1$, $\bar\Lambda = (\Lambda_{\max}+\Lambda_{\min})/2$ and the
anisotropy is $\alpha = (\Lambda_{\max}-\Lambda_{\min})/(\Lambda_{\max}+\Lambda_{\min})$.

### Why this makes a tissue converge and extend

The energy is the standard 2D vertex model,

$$E \;=\; \sum_{\text{cells}} \tfrac{K_A}{2}(A - A_0)^2 \;+\; \tfrac{K_P}{2}(P - P_0)^2
      \;+\; \sum_{\text{junctions}} \Lambda_{ij}\, \ell_{ij},$$

integrated by the stock `EulerSolver` with `auto_reconnect` on. Junctions aligned with
$\hat p$ carry the large $\Lambda$, so they shrink; once one falls below
`threshold_length` tyssue's `reconnect` fires a **T1**, and the new junction it mints is
oriented across $\hat p$ — where the tension is low, so it is free to grow. Each T1 moves
one cell width out of the polarity direction and into the perpendicular one. Repeated over
the sheet that is convergent extension, and it is driven by **cell rearrangement**, not by
cells stretching — a distinction the analysis at the end measures directly.

Note that the tension law is applied to the **live** geometry every step. A junction that
rotates towards the polarity axis is retensioned as it rotates, and a junction born of a T1
picks up its tension on the next step.


## Where the model lives — a process, and a reused behavior

`DirectionalLineTension` (`vivarium_tyssue/processes/regulations.py`) is a sibling of
`StochasticLineTension` and `AnisotropicTension`: it reads the epithelium through the
`datasets` port, computes the tensions **in the process**, and ships them as a
`unique_id -> tension` map through the `update_tension` behavior. It never touches the
mesh and `EulerSolver` is untouched by this experiment.

That split is the right one here — and it is the opposite of the choice `DifferentialAdhesion`
makes. There, whether a junction is heterotypic depends on the *pair of cells it separates*,
which a T1 rewires, so the classification has to happen on the live mesh inside the solver's
`EventManager`. Here a junction's tension depends only on **its own geometry**, which the
process reads straight out of `edge_df`, and the map is keyed by `unique_id`, so an entry
for a junction a T1 has since removed simply matches nothing.

A second behavior, the repo's generic `apply_gradient`, writes the per-edge alignment
$a(\theta)$ into an `edge_df` column (`polar_alignment`) so the polarity read-out lands in
the solver's `History` next to the tension. No new behavior was added for this experiment.

| piece | where it lives |
|---|---|
| `DirectionalLineTension` process | `vivarium_tyssue/processes/regulations.py` |
| `update_tension`, `apply_gradient` behaviors | `vivarium_tyssue/behaviors/behaviors.py` |
| process registration | `vivarium_tyssue/processes/__init__.py` |
| tests | `tests/test_directional_tension.py` |


## Setup


In [ ]:
from __future__ import annotations

import json
import sys
import time
import warnings
from copy import deepcopy
from pathlib import Path
from pprint import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection, PolyCollection

HERE = Path.cwd()
assert HERE.name == "directional_tension", (
    f"run this notebook with Experiments/directional_tension as the working "
    f"directory (currently {HERE})"
)
REPO = HERE.parents[1]
sys.path.insert(0, str(REPO))                    # vivarium_tyssue
sys.path.insert(0, str(REPO / "Experiments"))    # shared bigraph palette

DATA_DIR = HERE / "data"
OUT_DIR = HERE / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)
DATASET_NAME = "test_square.hf5"
RAW_MESH_PATH = DATA_DIR / DATASET_NAME          # the shared mesh, copied verbatim
MESH_PATH = DATA_DIR / "flat_sheet_2d.hf5"      # the same mesh, z dropped

COORDS = ["x", "y"]
FIG_DPI = 300
GIF_DPI = 110
SEED = 20260901

# Okabe-Ito: the three runs keep these colours everywhere.
RUN_COLORS = {"px": "#0072B2", "pxy": "#009E73", "control": "#999999"}

plt.rcParams.update({"figure.dpi": 110, "axes.grid": False,
                     "axes.spines.top": False, "axes.spines.right": False})

print("repo    :", REPO)
print("mesh    :", MESH_PATH)
print("outputs :", OUT_DIR)


## Step 1 — the mesh: the repo's shared flat sheet

The same `workspace/datasets/test_square.hf5` the **stochastic-tension** experiment runs
on, copied into `data/` on first use exactly as that experiment does. Reusing it keeps the
two flat-sheet experiments on one substrate, and it comes already `sanitize`d with
**trimmed borders** — the Voronoi border cells clipped rather than left ragged, so 43 of
the 206 cells are partial (four- and five-sided) and the sheet has a genuine free boundary.

One conversion: the stored mesh carries an all-zero `z` column, which makes tyssue infer a
3-D sheet. Dropping it gives a **true 2D** sheet (`dim == 2`, coords `x`/`y` only) so
`PlanarGeometry` applies and nothing in the run is quietly 2.5-D. The topology and the
vertex positions are untouched.

Because the border cells are clipped they start at about half the area of an interior cell.
$A_0$ and $P_0$ are set to the sheet **mean**, so the interior sits very close to its rest
state and the border half-cells relax outward slightly over the first few time units — a
transient the isotropic control measures too, which is why the CE index is normalised per
run at $t = 0$.


In [ ]:
import shutil

from tyssue import Sheet
from tyssue.config.geometry import planar_spec
from tyssue.geometry.planar_geometry import PlanarGeometry
from tyssue.io.hdf5 import load_datasets, save_datasets

SOURCE_MESH = REPO / "workspace" / "datasets" / DATASET_NAME


def ensure_dataset():
    '''Copy the shared flat sheet into (git-ignored) data/ on first use — the same
    mesh, and the same copy-on-first-use, as Experiments/stochastic_tension.'''
    if not RAW_MESH_PATH.exists():
        if not SOURCE_MESH.exists():
            raise FileNotFoundError(f"source mesh not found: {SOURCE_MESH}")
        shutil.copy(SOURCE_MESH, RAW_MESH_PATH)
        print(f"copied {SOURCE_MESH} -> {RAW_MESH_PATH}")
    return RAW_MESH_PATH


def to_planar(path):
    '''Load the sheet and drop its all-zero `z` so tyssue reads it as a true 2D sheet.

    Two subtleties. `Epithelium.__init__` re-materialises any column its *specs* declare,
    so the sheet has to be built on `planar_spec()` and the drop repeated afterwards, or
    `z` comes straight back at its default. And the drop has to reach `face`/`vert`
    (`edge` keeps a harmless `nz`, the planar normal), because dim is inferred from the
    stored columns — a leftover `z` anywhere makes `PlanarGeometry.update_all` fail on
    unpacking three coordinates into two.
    '''
    datasets = {
        key: df.drop(columns=[c for c in df.columns if c == "z"], errors="ignore")
        for key, df in load_datasets(str(path)).items()
    }
    sheet = Sheet("flat_sheet", datasets, specs=planar_spec(), coords=COORDS)
    for df in sheet.datasets.values():
        df.drop(columns=[c for c in df.columns if c == "z"], inplace=True, errors="ignore")
    PlanarGeometry.update_all(sheet)
    return sheet


ensure_dataset()
sheet = to_planar(RAW_MESH_PATH)

if MESH_PATH.exists():
    MESH_PATH.unlink()
save_datasets(str(MESH_PATH), sheet)

A_0 = float(sheet.face_df["area"].mean())
P_0 = float(sheet.face_df["perimeter"].mean())

interior = sheet.face_df["num_sides"] == 6
print(f"source             {SOURCE_MESH.relative_to(REPO)}")
print(f"dimension          {sheet.dim}D, coords {sheet.coords}")
print(f"cells / verts / half-edges   {sheet.Nf} / {sheet.Nv} / {sheet.Ne}")
print(f"polygon sides      {sheet.face_df['num_sides'].value_counts().sort_index().to_dict()}"
      f"   ({int((~interior).sum())} clipped border cells)")
print(f"cell area          mean {A_0:.4f}   interior {sheet.face_df.loc[interior, 'area'].mean():.4f}"
      f"   border {sheet.face_df.loc[~interior, 'area'].mean():.4f}")
print(f"cell perimeter     mean {P_0:.4f}")
print(f"mean edge length   {sheet.edge_df['length'].mean():.4f}"
      f"   (threshold_length below is set to ~1/5 of this)")
print(f"bounding box       {np.ptp(sheet.vert_df['x']):.2f} x {np.ptp(sheet.vert_df['y']):.2f}")
print(f"duplicate uids     "
      f"{ {e: int(sheet.datasets[e]['unique_id'].duplicated().sum()) for e in ('vert', 'edge', 'face')} }")
print(f"wrote              {MESH_PATH}")


The starting sheet, and the orientation of its junctions. The hexagonal tiling puts them
in essentially **three** orientation families — here 30°, 90° and 150°, with the trimmed
border contributing the small spread around them — which is worth keeping in mind when
reading the tension law below: with the polarity along $x$ the 30°/150° families are the
contractile ones and the 90° family sits at zero tension.


In [ ]:
def face_polygons(srce, trgt, face, verts):
    '''Ordered vertex polygon for every face, chained from its half-edges.

    The half-edges of a face form a closed cycle srce -> trgt, but nothing in the
    solver guarantees they stay in cycle order in `edge_df` after a T1, so the cycle
    is walked explicitly rather than trusting the row order.'''
    order = np.argsort(face, kind="stable")
    srce, trgt, face = srce[order], trgt[order], face[order]
    starts = np.searchsorted(face, np.unique(face))
    ends = np.r_[starts[1:], len(face)]
    polys = []
    for a, b in zip(starts, ends):
        nxt = dict(zip(srce[a:b], trgt[a:b]))
        v0 = srce[a]
        ring, v = [v0], nxt.get(v0)
        while v is not None and v != v0 and len(ring) <= (b - a):
            ring.append(v)
            v = nxt.get(v)
        polys.append(verts[ring])
    return polys


def draw_sheet(ax, verts, srce, trgt, face, *, edge_values=None, cmap="coolwarm",
               vlim=(0.0, 1.0), face_color="#ffffff", edge_color="#333333",
               lw=1.0, xlim=None, ylim=None):
    '''One 2D frame: filled cells, junctions optionally coloured by a per-edge value.

    Cells are outlined in light grey *underneath* the coloured junctions. Without that
    underlay, junctions sitting at whichever end of the colour scale is pale vanish into
    the fill and the tissue reads as a scatter of disconnected slivers even when the mesh
    is perfectly intact. The grey outline means cell shape is always legible and the
    colour only has to carry the tension.

    The colour map is *diverging* (`coolwarm`: blue slack, red taut) for the same reason.
    A sequential map spends one of its ends on near-white, and near-white is exactly what
    a junction perpendicular to the polarity axis gets — so the junctions the mechanism is
    about would be the ones you could not see.'''
    ax.add_collection(PolyCollection(
        face_polygons(srce, trgt, face, verts),
        facecolors=face_color, edgecolors="#9a9a9a",
        linewidths=0.5 if edge_values is not None else 0.0, zorder=0))
    segs = np.stack([verts[srce], verts[trgt]], axis=1)
    if edge_values is None:
        lc = LineCollection(segs, colors=edge_color, linewidths=lw, zorder=1)
    else:
        lc = LineCollection(segs, cmap=cmap, linewidths=lw, zorder=1)
        lc.set_array(np.asarray(edge_values))
        lc.set_clim(*vlim)
    ax.add_collection(lc)
    ax.set_aspect("equal")
    ax.set_xlim(*(xlim if xlim is not None else (verts[:, 0].min(), verts[:, 0].max())))
    ax.set_ylim(*(ylim if ylim is not None else (verts[:, 1].min(), verts[:, 1].max())))
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    return lc


V0 = sheet.vert_df[COORDS].to_numpy(float)
E0 = (sheet.edge_df["srce"].to_numpy(), sheet.edge_df["trgt"].to_numpy(),
      sheet.edge_df["face"].to_numpy())

fig, axes = plt.subplots(1, 2, figsize=(9.4, 4.6),
                         gridspec_kw={"width_ratios": [1.25, 1.0]})
draw_sheet(axes[0], V0, *E0, lw=0.8)
axes[0].set_title(f"$t = 0$: {sheet.Nf} cells, trimmed border, free boundary", fontsize=10)

theta0 = np.degrees(np.arctan2(sheet.edge_df["dy"], sheet.edge_df["dx"])) % 180.0
axes[1].hist(theta0, bins=np.arange(0, 181, 5), color="#0072B2")
axes[1].set(xlabel="junction orientation (deg, mod 180)", ylabel="half-edges",
            xticks=[0, 30, 60, 90, 120, 150, 180])
axes[1].set_title("three orientation families", fontsize=10)
for s in axes[1].spines.values():
    s.set_visible(True)
axes[1].spines["top"].set_visible(False); axes[1].spines["right"].set_visible(False)

fig.tight_layout()
fig.savefig(OUT_DIR / "initial_condition.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


## Step 2 — the tension law, drawn

Two ways of looking at the same equation. On the left, $\Lambda(\theta)$ for the three
profiles at the values used below ($\Lambda_{\min} = 0$, $\Lambda_{\max} = 0.6$); on the
right, the starting mesh with each junction coloured by the tension it is actually given
for polarity $(1, 0)$ and for polarity $(1, 1)$. The arrow is $\hat p$.

Read the right-hand panels as the prediction: the *dark* junctions are the ones that will
contract, and they are the ones lying along the arrow.


In [ ]:
from vivarium_tyssue.processes.regulations import ALIGNMENT_MAP

LAMBDA_MIN = 0.0
# Keep this small. It competes directly with the area term (K_A = 1, A_0 = 1), and a
# tension that wins that competition does not deform the tissue, it degrades it. See the
# measured trade-off table in the markdown above; 0.15 is the chosen compromise.
LAMBDA_MAX = 0.15
PROFILE = "cos2"
SHARPNESS = 1.0


def tension_law(theta, profile=PROFILE, sharpness=SHARPNESS,
                lo=LAMBDA_MIN, hi=LAMBDA_MAX):
    '''Lambda(theta) — the same expression DirectionalLineTension.update evaluates.'''
    return lo + (hi - lo) * ALIGNMENT_MAP[profile](theta) ** sharpness


def edge_tension(verts, srce, trgt, polarity, **kwds):
    '''Per-half-edge tension for a frame, from the acute angle to `polarity`.'''
    p = np.asarray(polarity, float)
    p = p / np.linalg.norm(p)
    d = verts[trgt] - verts[srce]
    length = np.linalg.norm(d, axis=1)
    cos_t = np.zeros(len(d))
    good = length > 0
    cos_t[good] = np.abs(d[good] @ p) / length[good]
    return tension_law(np.arccos(np.clip(cos_t, 0.0, 1.0)), **kwds)


POLARITIES = {"px": (1.0, 0.0), "pxy": (1.0, 1.0)}

fig = plt.figure(figsize=(11.0, 4.0))
gs = fig.add_gridspec(1, 3, width_ratios=[1.15, 1.0, 1.0], wspace=0.25)

ax = fig.add_subplot(gs[0, 0])
th = np.linspace(0, np.pi / 2, 401)
for name, style in zip(["cos2", "abs_cos", "linear"], ["-", "--", ":"]):
    ax.plot(np.degrees(th), tension_law(th, profile=name), style, lw=2, label=name)
ax.plot(np.degrees(th), tension_law(th, sharpness=4.0), "-", lw=1.2, color="#CC79A7",
        label="cos2, sharpness 4")
ax.set(xlabel=r"acute angle $\theta$ to polarity (deg)", ylabel=r"line tension $\Lambda$",
       xticks=[0, 15, 30, 45, 60, 75, 90], ylim=(-0.03, LAMBDA_MAX * 1.08))
ax.axhline(LAMBDA_MAX, color="k", lw=0.6, ls="-.")
ax.axhline(LAMBDA_MIN, color="k", lw=0.6, ls="-.")
ax.legend(frameon=False, fontsize=8)
ax.set_title(r"$\Lambda(\theta)$: parallel is strongest", fontsize=10)
for s in ax.spines.values():
    s.set_visible(True)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

pad = 0.6
xlim0 = (V0[:, 0].min() - pad, V0[:, 0].max() + pad)
ylim0 = (V0[:, 1].min() - pad, V0[:, 1].max() + pad)
for col, (name, pol) in enumerate(POLARITIES.items()):
    axm = fig.add_subplot(gs[0, col + 1])
    lc = draw_sheet(axm, V0, *E0, lw=1.4, xlim=xlim0, ylim=ylim0,
                    edge_values=edge_tension(V0, E0[0], E0[1], pol),
                    vlim=(LAMBDA_MIN, LAMBDA_MAX))
    c = V0.mean(axis=0)
    u = np.asarray(pol, float); u = u / np.linalg.norm(u) * 4.0
    axm.annotate("", xy=c + u, xytext=c - u,
                 arrowprops=dict(arrowstyle="-|>", lw=2.2, color="#009E73"))
    axm.set_title(rf"$\hat p \propto$ {tuple(int(v) for v in pol)}", fontsize=10)

cb = fig.colorbar(lc, ax=fig.axes[1:], fraction=0.03, pad=0.02)
cb.set_label(r"$\Lambda$")
fig.savefig(OUT_DIR / "tension_law.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


## Step 3 — the parameters, set explicitly

$A_0$ and $P_0$ are taken from the starting patch, so the sheet begins at the minimum of
its area and perimeter terms and **every** subsequent deformation is paid for by the
tension term. That makes the isotropic control a genuine null: with a uniform $\Lambda$
the tissue only relaxes slightly and nothing rearranges.

`threshold_length` is the junction length at which tyssue's `reconnect` fires a T1. At
0.12 it is ~1/5 of the starting junction length: short enough that only genuinely
collapsing junctions trigger, long enough that they trigger before the mesh degenerates.

$\Lambda_{\max}$ is the parameter to be careful with, because it competes directly with the
area term ($K_A = 1$, $A_0 = 1$) and a tension that wins that competition does not deform
the tissue, it *degrades* it. Measured at $t = 240$, sweeping $\Lambda_{\max}$ with
everything else held fixed:

| $\Lambda_{\max}$ | CE index | area kept | smallest cell | mean sides | verdict |
|---|---|---|---|---|---|
| 0.05 | 1.29 | 96 % | 1.01 | 5.72 | pristine, modest signal |
| 0.10 | 1.62 | 93 % | 0.97 | 5.71 | very clean |
| **0.15** | **1.96** | **90 %** | **0.92** | **5.70** | **chosen** |
| 0.20 | 2.36 | 87 % | 0.81 | 5.68 | intact, cells noticeably squashed |
| 0.30 | 3.27 | 81 % | 0.68 | 5.56 | starting to degrade |

0.15 is the compromise: a clear convergent-extension signal, with the area loss saturating
early (most of it by $t \approx 40$) rather than running away, no cell ending below 0.92 of
the sheet's rest area, and the polygon distribution essentially unchanged. Raising it buys
signal and spends tissue integrity at roughly even rates; by 0.3 cells are down to two
thirds of their rest area. (This sheet tolerates tension noticeably better than a
perfectly regular hexagonal patch does — the trimmed border gives it somewhere to yield.)


In [ ]:
AREA_ELASTICITY = 1.0        # K_A
PREFERED_AREA = A_0          # A_0, the patch's own cell area
PERIMETER_ELASTICITY = 0.1   # K_P
PREFERED_PERIMETER = P_0     # P_0, the patch's own cell perimeter
VISCOSITY = 1.0              # eta

# --- the polarity law -------------------------------------------------------
# LAMBDA_MIN / LAMBDA_MAX / PROFILE / SHARPNESS were set in Step 2.
# The control uses this single tension on every junction, whatever its angle: the
# mean of the two extremes, so the two runs put in the same *total* contractility.
LAMBDA_CONTROL = 0.5 * (LAMBDA_MIN + LAMBDA_MAX)

# --- topology (T1 transitions) ---------------------------------------------
THRESHOLD_LENGTH = 0.12      # junctions shorter than this merge their two vertices
P_4 = 5.0                    # per unit time: a rank-4 vertex detaches again
P_5P = 0.5                   # per unit time: a rank-5+ vertex detaches

# --- integration ------------------------------------------------------------
DT = 0.1                     # solver / process interval; 0.05 gives the same
                             # trajectory to ~1%, so this is the converged step
TF = 240.0                   # total simulated time (TF / DT = 2400 steps)

params = pd.Series({
    "K_A  area_elasticity": AREA_ELASTICITY,
    "A_0  prefered_area": PREFERED_AREA,
    "K_P  perimeter_elasticity": PERIMETER_ELASTICITY,
    "P_0  prefered_perimeter": PREFERED_PERIMETER,
    "L_min  tension, perpendicular": LAMBDA_MIN,
    "L_max  tension, parallel": LAMBDA_MAX,
    "L_ctrl  tension, control": LAMBDA_CONTROL,
    "profile": PROFILE,
    "n  sharpness": SHARPNESS,
    "eta  viscosity": VISCOSITY,
    "l_min  threshold_length": THRESHOLD_LENGTH,
    "p_4": P_4,
    "p_5p": P_5P,
    "dt": DT,
    "t_f": TF,
}, name="value")
params.to_frame()


## Step 4 — the `EulerSolver` configuration

`PlanarGeometry` and the plain `model_factory`: a genuinely two-dimensional sheet, no
height, no lumen, no out-of-plane term. `auto_reconnect` is **essential** — convergent
extension *is* T1 transitions, and with it off the sheet can only stretch elastically.

`polar_alignment` is declared in the `edge_df` parameters so that the column exists when
the solver builds its `History`; the `apply_gradient` behavior fills it every step. A
column that does not exist at build time is never recorded, however faithfully it is
written later.


In [ ]:
tyssue_config = {
    "name": "Directional Tension Patch",
    "eptm": str(MESH_PATH),
    "tissue_type": "Sheet",

    "parameters": {
        "face_df": {
            "area_elasticity": AREA_ELASTICITY,
            "prefered_area": PREFERED_AREA,
            "perimeter_elasticity": PERIMETER_ELASTICITY,
            "prefered_perimeter": PREFERED_PERIMETER,
            "is_alive": 1.0,
        },
        "edge_df": {
            # Seed only — DirectionalLineTension owns this column from step 1 on.
            "line_tension": 0.0,
            "is_active": 1.0,
            # Must exist now so History records it (see the note above).
            "polar_alignment": 0.0,
        },
        "vert_df": {
            "viscosity": VISCOSITY,
            "is_active": 1.0,
        },
    },

    "geom": "PlanarGeometry",
    "effectors": ["LineTension", "FaceAreaElasticity", "PerimeterElasticity"],
    "ref_effector": "FaceAreaElasticity",
    "factory": "model_factory",

    "settings": {
        "threshold_length": THRESHOLD_LENGTH,
        "p_4": P_4,
        "p_5p": P_5P,
    },

    "auto_reconnect": True,          # T1 transitions — no convergent extension without them
    "check_intersections": False,
    "intersection_options": {},
    "bounds": {},
    "output_columns": {},

    # Only what the movies and the analysis need. `unique_id` is what the T1 metric
    # tracks cells by: the row index is renumbered by every reset_index.
    "history_columns": {
        "vert_df": [],
        "edge_df": ["unique_id", "line_tension", "polar_alignment", "length"],
        "face_df": ["unique_id", "area", "perimeter", "num_sides"],
    },

    "maps": {},
    "backend": "python",             # PlanarGeometry is outside the Rust kernels
    "substeps": 1,
    "max_displacement": 0.0,
    "record_history": True,
}

pprint(tyssue_config, sort_dicts=False, width=96)


## Step 5 — the `DirectionalLineTension` configuration

Three runs, differing only in the polarity law:

| run | `polarity` | `tension_min` | `tension_max` |
|---|---|---|---|
| `px` | $(1, 0)$ | 0.0 | 0.15 |
| `pxy` | $(1, 1)$ — *not* normalised, and not a lattice axis | 0.0 | 0.15 |
| `control` | $(1, 0)$ | 0.075 | 0.075 |

The control still runs the same process on the same wiring; setting the two extremes
equal makes $\Lambda(\theta)$ constant, so the polarity vector becomes irrelevant and the
tension is isotropic. That isolates *anisotropy* as the cause — not the presence of line
tension, and not the total contractility, which is matched.


In [ ]:
def polarity_config(polarity, tension_min=LAMBDA_MIN, tension_max=LAMBDA_MAX):
    return {
        "polarity": list(polarity),   # normalised by the process; any non-zero vector
        "tension_min": tension_min,
        "tension_max": tension_max,
        "profile": PROFILE,
        "sharpness": SHARPNESS,
        "coords": [],                 # empty -> inferred from len(polarity)
        "record_column": "polar_alignment",
    }


RUNS = {
    "px":      dict(label=r"$\hat p \parallel (1,0)$",  polarity=POLARITIES["px"],
                    config=polarity_config(POLARITIES["px"])),
    "pxy":     dict(label=r"$\hat p \parallel (1,1)$",  polarity=POLARITIES["pxy"],
                    config=polarity_config(POLARITIES["pxy"])),
    "control": dict(label="isotropic control",          polarity=POLARITIES["px"],
                    config=polarity_config(POLARITIES["px"], LAMBDA_CONTROL, LAMBDA_CONTROL)),
}

pd.DataFrame({k: v["config"] for k, v in RUNS.items()}).T


## Step 6 — the composite spec

Two processes on one interval. `EulerSolver` publishes the epithelium to the
`Tissue State` store; `DirectionalLineTension` reads it, and writes back into the
`Behaviors` store the solver drains at the end of its next step. The solver's own
`behaviors_update` output is what clears that store once the behaviors have run.


In [ ]:
def build_spec(run_key: str) -> dict:
    return {
        "Tyssue": {
            "_type": "process",
            "address": "local:EulerSolver",
            "config": deepcopy(tyssue_config),
            "inputs": {
                "behaviors": ["Behaviors"],
                "global_time": ["global_time"],
            },
            "outputs": {
                "datasets": ["Tissue State"],
                "network_changed": ["Network Changed"],
                "behaviors_update": ["Behaviors"],
            },
            "interval": DT,
        },
        "Polarity": {
            "_type": "process",
            "address": "local:DirectionalLineTension",
            "config": deepcopy(RUNS[run_key]["config"]),
            "inputs": {"datasets": ["Tissue State"]},
            "outputs": {"behaviors": ["Behaviors"]},
            "interval": DT,
        },
        "Network Changed": False,
        "Behaviors": {},
    }


spec = build_spec("px")
pprint({k: (v if k != "Tyssue" else {**v, "config": "<see above>"})
        for k, v in spec.items()}, sort_dicts=False, width=96)


## Step 7 — the bigraph


In [ ]:
from bigraph_style import plot_experiment_bigraph
from vivarium_tyssue.core import build_core

core = build_core()   # the core the simulations themselves run on

plot_experiment_bigraph(
    spec, core=core,
    filename="directional_tension_bigraph",
    out_dir=OUT_DIR,
)


## Step 8 — run

Three runs of 2400 steps each, a couple of minutes apiece. The T1 detachment rule
(`p_4` / `p_5p`) is stochastic, so the seed is reset before each.


In [ ]:
from process_bigraph import Composite

histories = {}
for key, run in RUNS.items():
    np.random.seed(SEED)
    sim = Composite({"state": build_spec(key)}, core=core)
    t0 = time.time()
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        sim.run(TF)
    history = sim.state["Tyssue"]["instance"].history
    history.update_datasets()
    eptm = sim.state["Tyssue"]["instance"].eptm
    histories[key] = history
    print(f"[{key:>7}] {time.time() - t0:5.0f}s  "
          f"{len(list(history.time_stamps))} frames  "
          f"Nf={eptm.Nf} Nv={eptm.Nv} Ne={eptm.Ne}")


The mesh has to survive the run for any of the numbers below to mean anything: no cell
lost, every half-edge still paired with an opposite except on the free boundary, and no
degenerate (fewer than three sided) face.


In [ ]:
def mesh_report(history, label):
    edge = history.datasets["edge"]
    last = edge[edge["time"] == edge["time"].max()]
    face = history.datasets["face"]
    face_last = face[face["time"] == face["time"].max()]
    pairs = pd.MultiIndex.from_arrays([last["srce"], last["trgt"]])
    print(f"[{label:>7}] cells {face_last['face'].nunique():4d}  "
          f"half-edges {len(last):5d}  "
          f"duplicated (srce,trgt) {int(pairs.duplicated().sum()):3d}  "
          f"faces with < 3 sides {int((face_last['num_sides'] < 3).sum()):3d}  "
          f"min junction length {last['length'].min():.4f}")


for key, history in histories.items():
    mesh_report(history, key)


## Step 9 — the movies

**The axes are sized to the largest frame, not the first.** The tissue grows along the
extension axis as it runs — the `px` sheet ends ~30 % longer in $y$ than it started — so
limits taken from frame 0 (which is what tyssue's own `create_gif` does) would crop the
end of the movie and, worse, make the extension invisible by rescaling it away.

`frame_bounds` therefore scans **every** frame of **every** run, takes the union of the
bounding boxes, and then squares that box up to the figure's aspect ratio so the two
directions stay on the same scale. Combined with a fixed `figsize`, a fixed `dpi` and
*no* `bbox_inches="tight"` (which would re-crop each frame independently), that keeps
one unit of $x$ the same number of pixels as one unit of $y$ in every frame of all three
movies — so the three are directly comparable, and the shape change on screen is the real
shape change.


In [ ]:
from PIL import Image

GIF_FRAMES = 90
GIF_FIGSIZE = (5.2, 5.2)
GIF_MARGIN = 0.04          # fraction of the larger span, added on every side


def frame_bounds(hists, margin=GIF_MARGIN, aspect=GIF_FIGSIZE[0] / GIF_FIGSIZE[1]):
    '''(xlim, ylim) enclosing every frame of every history, at the figure's aspect.'''
    lo = np.full(2, np.inf)
    hi = np.full(2, -np.inf)
    for h in hists:
        v = h.datasets["vert"]
        g = v.groupby("time")[COORDS]
        lo = np.minimum(lo, g.min().to_numpy().min(axis=0))
        hi = np.maximum(hi, g.max().to_numpy().max(axis=0))
    pad = margin * (hi - lo).max()
    lo, hi = lo - pad, hi + pad
    centre, span = (lo + hi) / 2, hi - lo
    # Grow the short side so span_x / span_y == aspect: never crop, only add room.
    if span[0] / span[1] < aspect:
        span[0] = span[1] * aspect
    else:
        span[1] = span[0] / aspect
    return ((centre[0] - span[0] / 2, centre[0] + span[0] / 2),
            (centre[1] - span[1] / 2, centre[1] + span[1] / 2))


def first_active_time(history):
    '''First recorded time whose line_tension the process has actually written.

    The solver records frame k, the process reads it and emits, and the behavior lands
    on frame k+1 — so the opening frames still carry the seed value of 0 and would paint
    the whole sheet one flat colour. Find the first frame that does not.'''
    edge = history.datasets["edge"]
    for t, e in edge.groupby("time"):
        if float(e["line_tension"].abs().max()) > 0:
            return float(t)
    return float(edge["time"].min())


# A run whose two tension extremes are equal has a uniform line tension: colouring the
# junctions by it says nothing, and on a diverging map its single value lands on the
# neutral midpoint and disappears. Draw those runs with plain dark junctions instead.
IS_UNIFORM = {k: run["config"]["tension_min"] == run["config"]["tension_max"]
              for k, run in RUNS.items()}

XLIM, YLIM = frame_bounds(histories.values())
print(f"global frame bounds  x {XLIM[0]:.2f} .. {XLIM[1]:.2f}   "
      f"y {YLIM[0]:.2f} .. {YLIM[1]:.2f}")
print(f"starting patch       x {V0[:, 0].min():.2f} .. {V0[:, 0].max():.2f}   "
      f"y {V0[:, 1].min():.2f} .. {V0[:, 1].max():.2f}")


def render_gif(history, out_path, label, polarity, n_frames=GIF_FRAMES,
               duration=90, xlim=None, ylim=None, uniform=False):
    '''Junctions coloured by their live line_tension, on fixed global axes.'''
    xlim = XLIM if xlim is None else xlim
    ylim = YLIM if ylim is None else ylim
    vert = history.datasets["vert"]
    edge = history.datasets["edge"]
    times = np.array(sorted(vert["time"].unique()))
    start = int(np.searchsorted(times, first_active_time(history)))
    idx = np.unique(np.round(np.linspace(start, len(times) - 1, n_frames)).astype(int))
    vert_by_t = dict(tuple(vert.groupby("time")))
    edge_by_t = dict(tuple(edge.groupby("time")))

    u = np.asarray(polarity, float)
    u = u / np.linalg.norm(u) * 0.09 * (xlim[1] - xlim[0])
    anchor = np.array([xlim[0] + 0.13 * (xlim[1] - xlim[0]),
                       ylim[0] + 0.07 * (ylim[1] - ylim[0])])

    frames = []
    for i in idx:
        t = times[i]
        v = vert_by_t[t].sort_values("vert")
        e = edge_by_t[t]
        pos = np.empty((int(v["vert"].max()) + 1, 2))
        pos[v["vert"].to_numpy()] = v[COORDS].to_numpy(float)

        fig, ax = plt.subplots(figsize=GIF_FIGSIZE)
        lc = draw_sheet(ax, pos, e["srce"].to_numpy(), e["trgt"].to_numpy(),
                        e["face"].to_numpy(), lw=1.3,
                        edge_values=None if uniform else e["line_tension"].to_numpy(),
                        vlim=(LAMBDA_MIN, LAMBDA_MAX), xlim=xlim, ylim=ylim)
        ax.annotate("", xy=anchor + u, xytext=anchor - u,
                    arrowprops=dict(arrowstyle="-|>", lw=2.0, color="#009E73"))
        ax.set_title(f"{label}    $t = {t:6.2f}$", fontsize=11)
        if uniform:
            # Keep the frame geometry identical to the coloured runs (the movies share
            # one set of axes), so reserve the colourbar's room and leave it empty.
            fig.colorbar(lc, ax=ax, fraction=0.035, pad=0.02).ax.set_visible(False)
        else:
            cb = fig.colorbar(lc, ax=ax, fraction=0.035, pad=0.02)
            cb.set_label(r"line tension $\Lambda$", fontsize=9)
            cb.ax.tick_params(labelsize=8)

        fig.canvas.draw()
        # No bbox_inches="tight": every frame must keep the same pixel geometry.
        buf = np.asarray(fig.canvas.buffer_rgba())
        frames.append(Image.fromarray(buf[..., :3].copy()))
        plt.close(fig)

    sizes = {f.size for f in frames}
    assert len(sizes) == 1, f"frames differ in size: {sizes}"
    frames[0].save(out_path, save_all=True, append_images=frames[1:],
                   duration=duration, loop=0, optimize=True)
    print(f"wrote {out_path}  ({len(frames)} frames, {sizes.pop()} px, "
          f"{out_path.stat().st_size / 1e6:.1f} MB)")
    return out_path


plt.rcParams["figure.dpi"] = GIF_DPI
gif_paths = {}
for key, run in RUNS.items():
    gif_paths[key] = render_gif(histories[key], OUT_DIR / f"{key}.gif",
                                run["label"], run["polarity"],
                                uniform=IS_UNIFORM[key])
plt.rcParams["figure.dpi"] = 110


In [ ]:
from IPython.display import Image as IPyImage, display

for key in RUNS:
    print(RUNS[key]["label"])
    display(IPyImage(filename=str(gif_paths[key])))


Start and end, side by side, all six panels on the same axes as the movies — so the
`px` patch really is that much narrower and that much longer, and the control really is
that unchanged.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11.4, 8.0))
for col, (key, run) in enumerate(RUNS.items()):
    h = histories[key]
    times = sorted(h.datasets["vert"]["time"].unique())
    for row, t in enumerate((first_active_time(h), times[-1])):
        v = h.datasets["vert"]
        v = v[v["time"] == t].sort_values("vert")
        e = h.datasets["edge"]
        e = e[e["time"] == t]
        pos = np.empty((int(v["vert"].max()) + 1, 2))
        pos[v["vert"].to_numpy()] = v[COORDS].to_numpy(float)
        draw_sheet(axes[row, col], pos, e["srce"].to_numpy(), e["trgt"].to_numpy(),
                   e["face"].to_numpy(), lw=0.7,
                   edge_values=None if IS_UNIFORM[key] else e["line_tension"].to_numpy(),
                   vlim=(LAMBDA_MIN, LAMBDA_MAX), xlim=XLIM, ylim=YLIM)
        axes[row, col].set_title(f"{run['label']},  $t = {t:g}$", fontsize=10)
fig.tight_layout()
fig.savefig(OUT_DIR / "start_end.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


## Step 10 — measuring the convergent extension

Everything below is recomputed from the recorded mesh (positions + topology), never
read back off a column the process wrote — so the metrics are an independent check on
the model rather than a restatement of it.

**Tissue shape.** Project every vertex onto the polarity axis $\hat p$ and onto
$\hat p^{\perp}$, and take twice the standard deviation of each as the tissue's width
$L_{\parallel}$ and length $L_{\perp}$. (Second moments rather than the bounding box: a
single vertex flicking out on the free boundary moves the box and barely moves the
moment.) The **convergent-extension index** is their ratio, normalised to 1 at $t = 0$:

$$\mathrm{CE}(t) \;=\; \frac{L_{\perp}(t) / L_{\parallel}(t)}{L_{\perp}(0) / L_{\parallel}(0)}$$

**Cell shape.** The same second-moment construction per cell gives each cell an
elongation $\sqrt{\lambda_1/\lambda_2}$ and a long-axis direction; averaging
nematically against $\hat p$ gives $S_{\text{cell}}$, positive when cells are elongated
*along* the polarity axis and negative when they are elongated *across* it.

**T1 transitions**, counted as *completed* rearrangements: a pair of cells that becomes
adjacent having never been adjacent before. A junction that shrinks and then re-grows
between the same two cells exchanged no neighbours and is not counted.


In [ ]:
def _opposite_positional(srce, trgt):
    '''Positional index of each half-edge's opposite, -1 where there is none.'''
    forward = pd.Series(np.arange(len(srce)),
                        index=pd.MultiIndex.from_arrays([srce, trgt]))
    forward = forward[~forward.index.duplicated(keep="first")]
    opposite = forward.reindex(pd.MultiIndex.from_arrays([trgt, srce])).to_numpy()
    return np.where(np.isnan(opposite), -1, np.nan_to_num(opposite)).astype(int)


def _second_moment_axes(cxx, cxy, cyy):
    '''Eigenvalues (l1 >= l2) and long-axis angle of a batch of 2x2 covariances.'''
    tr, det = cxx + cyy, cxx * cyy - cxy ** 2
    disc = np.sqrt(np.maximum(tr ** 2 / 4.0 - det, 0.0))
    l1, l2 = tr / 2.0 + disc, tr / 2.0 - disc
    return l1, l2, 0.5 * np.arctan2(2.0 * cxy, cxx - cyy)


def run_metrics(history, polarity, stride=4):
    '''Per-frame tissue shape, cell shape, junction nematics and completed T1s.'''
    p = np.asarray(polarity, float)
    p = p / np.linalg.norm(p)
    q = np.array([-p[1], p[0]])
    phi_p = np.arctan2(p[1], p[0])

    vert_by_t = dict(tuple(history.datasets["vert"].groupby("time")))
    face_by_t = dict(tuple(history.datasets["face"].groupby("time")))

    seen, n_t1, rows = None, 0, []
    for i, (t, e) in enumerate(history.datasets["edge"].groupby("time")):
        v = vert_by_t[t].sort_values("vert")
        pos = np.empty((int(v["vert"].max()) + 1, 2))
        pos[v["vert"].to_numpy()] = v[COORDS].to_numpy(float)
        srce, trgt = e["srce"].to_numpy(), e["trgt"].to_numpy()

        # --- completed T1s: pairs of cells newly adjacent (every frame, no stride) ---
        face_uid = face_by_t[t].set_index("face")["unique_id"]
        opp = _opposite_positional(srce, trgt)
        interior = opp >= 0
        own = face_uid.reindex(e["face"].to_numpy()).to_numpy()
        other = own[np.where(interior, opp, 0)]
        pairs = {frozenset(pr) for pr in
                 zip(own[interior].astype(int), other[interior].astype(int))}
        if seen is None:
            seen = set(pairs)
        else:
            n_t1 += len(pairs - seen)
            seen |= pairs

        if i % stride:
            continue

        # --- tissue shape, in the polarity frame ---
        used = np.unique(np.r_[srce, trgt])
        xy = pos[used] - pos[used].mean(axis=0)
        L_par, L_perp = 2 * (xy @ p).std(), 2 * (xy @ q).std()

        # --- cell shape: second moments of each face's vertex ring ---
        d = pd.DataFrame({"face": e["face"].to_numpy(),
                          "x": pos[srce, 0], "y": pos[srce, 1]})
        d["xx"], d["xy"], d["yy"] = d.x * d.x, d.x * d.y, d.y * d.y
        m = d.groupby("face").mean()
        l1, l2, ang = _second_moment_axes(m.xx - m.x ** 2,
                                          m.xy - m.x * m.y,
                                          m.yy - m.y ** 2)
        ok = l2 > 1e-12
        elong = np.sqrt(l1[ok] / l2[ok])
        nem = (l1 - l2) / (l1 + l2)
        s_cell = float(np.mean(nem[ok] * np.cos(2 * (ang[ok] - phi_p))))

        # --- junction nematics, length-weighted ---
        dvec = pos[trgt] - pos[srce]
        length = np.linalg.norm(dvec, axis=1)
        phi = np.arctan2(dvec[:, 1], dvec[:, 0])
        w = length / length.sum()
        s_junction = float(np.sum(w * np.cos(2 * (phi - phi_p))))
        cos_t = np.divide(np.abs(dvec @ p), length,
                          out=np.zeros_like(length), where=length > 0)
        alignment = ALIGNMENT_MAP[PROFILE](np.arccos(np.clip(cos_t, 0, 1)))

        rows.append({
            "time": float(t),
            "L_parallel": float(L_par),
            "L_perp": float(L_perp),
            "aspect": float(L_perp / L_par),
            "tissue_area": float(face_by_t[t]["area"].sum()),
            # each interior junction is two half-edges; a boundary one is a single
            "n_junctions": int(interior.sum()) // 2 + int((~interior).sum()),
            "n_cells": int(face_by_t[t]["face"].nunique()),
            "n_t1": int(n_t1),
            "cell_elongation": float(np.mean(elong)),
            "S_cell": s_cell,
            "S_junction": s_junction,
            "mean_alignment": float(np.sum(w * alignment)),
            "mean_junction_length": float(length.mean()),
        })

    out = pd.DataFrame(rows)
    out["ce_index"] = out["aspect"] / out["aspect"].iloc[0]
    # Cell aspect-ratio change, and the share of the tissue's shape change that the
    # cells' own shape change does NOT account for (1 = pure rearrangement, 0 = affine).
    out["cell_aspect_change"] = out["cell_elongation"] / out["cell_elongation"].iloc[0]
    with np.errstate(divide="ignore", invalid="ignore"):
        out["rearrangement_share"] = 1.0 - np.log(out["cell_aspect_change"]) / np.log(
            out["ce_index"].where(out["ce_index"] != 1.0))
    return out


STRIDE = 6   # ~400 sampled points out of 2400 frames; T1s are counted on every frame
metrics = {}
for key, run in RUNS.items():
    t0 = time.time()
    metrics[key] = run_metrics(histories[key], run["polarity"], stride=STRIDE)
    metrics[key].insert(0, "run", key)
    print(f"[{key:>7}] {len(metrics[key])} sampled frames in {time.time() - t0:.0f}s")

all_metrics = pd.concat(metrics.values(), ignore_index=True)
all_metrics.to_csv(OUT_DIR / "metrics.csv", index=False)
all_metrics.groupby("run").tail(1).set_index("run")[
    ["time", "L_parallel", "L_perp", "ce_index", "n_t1", "cell_elongation",
     "cell_aspect_change", "rearrangement_share", "S_cell", "S_junction", "tissue_area"]
].round(3)


### The headline: convergence, extension, and the index that combines them


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.6, 3.9))

for key, run in RUNS.items():
    m, c = metrics[key], RUN_COLORS[key]
    axes[0].plot(m.time, m.L_parallel / m.L_parallel.iloc[0], color=c, lw=2,
                 label=run["label"])
    axes[0].plot(m.time, m.L_perp / m.L_perp.iloc[0], color=c, lw=2, ls="--")
    axes[1].plot(m.time, m.ce_index, color=c, lw=2, label=run["label"])
    axes[2].plot(m.time, m.n_t1, color=c, lw=2, label=run["label"])

axes[0].axhline(1.0, color="k", lw=0.6)
axes[0].set(xlabel="$t$", ylabel="length / length at $t=0$")
axes[0].set_title(r"solid: $L_\parallel$ (converges)" "\n" r"dashed: $L_\perp$ (extends)",
                  fontsize=10)
axes[0].legend(frameon=False, fontsize=8, loc="center left")

axes[1].axhline(1.0, color="k", lw=0.6)
axes[1].set(xlabel="$t$", ylabel="CE index")
axes[1].set_title(r"$\mathrm{CE}(t) = (L_\perp/L_\parallel)\,/\,(L_\perp/L_\parallel)|_0$",
                  fontsize=10)
axes[1].legend(frameon=False, fontsize=8)

axes[2].set(xlabel="$t$", ylabel="completed T1 transitions")
axes[2].set_title("cell rearrangements", fontsize=10)
axes[2].legend(frameon=False, fontsize=8)

for ax in axes:
    ax.spines["left"].set_visible(True); ax.spines["bottom"].set_visible(True)
fig.tight_layout()
fig.savefig(OUT_DIR / "convergent_extension.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


### How much of it is rearrangement, and how much is cells changing shape?

A tissue can converge and extend two ways: its cells can **rearrange** past one another
(T1s), or they can simply **deform**, each cell elongating across the polarity axis with
the tissue. The textbook convergent-extension picture is the first. Both are happening
here, and it is worth measuring which dominates rather than assuming.

Compare the tissue's aspect-ratio change $\mathrm{CE}(t)$ with the cells' own,
$C(t) = \langle\sqrt{\lambda_1/\lambda_2}\rangle(t) \,/\, \langle\sqrt{\lambda_1/\lambda_2}\rangle(0)$,
as a **rearrangement share**

$$R(t) \;=\; 1 \;-\; \frac{\ln C(t)}{\ln \mathrm{CE}(t)}.$$

$R = 1$ is convergent extension carried entirely by rearrangement, with cells keeping
their shape; $R = 0$ is a purely affine deformation, cells changing shape exactly as much
as the tissue does.

**The two polarity directions land in genuinely different places**, which is the most
interesting thing in this notebook:

* $\hat p \parallel (1,0)$ — $R \approx 0.04$, only **22** T1s. Essentially affine: the
  cells stretch by as much as the tissue does ($\times 1.90$ against $\mathrm{CE} = 1.96$)
  and rearrangement contributes almost nothing.
* $\hat p \parallel (1,1)$ — $R \approx 0.63$, **247** T1s. Nearly two thirds of the
  tissue's shape change is carried by cells changing neighbours: the tissue reaches
  $\mathrm{CE} = 1.82$ while its cells elongate by only $\times 1.25$. This is convergent
  extension by intercalation, the textbook mechanism.

The cause is the lattice. The hexagonal tiling has its junctions in essentially three
orientation families (30°, 90°, 150° — see Step 1). With $\hat p$ along $x$ the 90° family
sits at $\theta = 90°$, where $\Lambda = \Lambda_{\min} = 0$: those junctions are not just
weak, they are *free*, so the sheet's cheapest response is simply to stretch them and
almost no junction ever gets short enough to fire a T1. Rotate $\hat p$ to 45° and the three
families land at $\theta = 15°, 45°, 75°$ — **none** of them free — so contraction cannot be
relieved by stretching one family, junctions do collapse, and the tissue rearranges instead.
Eleven times as many T1s, and a rearrangement share that goes from ~0 to 0.63.

So "convergent extension by cell rearrangement" is not a property of the tension law alone:
it depends on whether the polarity axis is commensurate with the tissue's own junction
orientations. Raising $\Lambda_{\min}$ off zero is the knob that removes the free family
and would push the $(1,0)$ case towards the $(1,1)$ one.

$S_{\text{cell}}$ shows the same thing more directly: it goes and stays negative, i.e. the
cells' long axes end up across the polarity axis. Meanwhile $S_{\text{junction}}$ and the
length-weighted alignment $\langle a(\theta)\rangle_\ell$ fall as the parallel
(high-tension) junctions are spent and the perpendicular ones grow — the junction network
really is being rebuilt, which is what the T1 count counts.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16.4, 3.9))
for key, run in RUNS.items():
    m, c = metrics[key], RUN_COLORS[key]
    axes[0].plot(m.ce_index, m.cell_aspect_change, color=c, lw=2, label=run["label"])
    axes[1].plot(m.time, m.rearrangement_share, color=c, lw=2, label=run["label"])
    axes[2].plot(m.time, m.S_cell, color=c, lw=2, label=run["label"])
    axes[3].plot(m.time, m.mean_alignment, color=c, lw=2, label=run["label"])

lim = (0.9, max(1.1, float(all_metrics.ce_index.max()) * 1.05))
axes[0].plot(lim, lim, "k--", lw=0.9, label="affine ($R = 0$)")
axes[0].set(xlim=lim, ylim=lim)
axes[1].axhline(0.0, color="k", lw=0.6, ls="--")
axes[1].axhline(1.0, color="k", lw=0.6, ls="--")
axes[1].set_ylim(-0.6, 1.1)
axes[2].axhline(0.0, color="k", lw=0.6)

for ax, (xlab, ylab, title) in zip(axes, [
        (r"tissue: $\mathrm{CE}$", r"cells: $C$", "cell shape vs. tissue shape\n(on the dashed line = affine)"),
        ("$t$", "$R$", "rearrangement share\n(1 = pure T1, 0 = pure deformation)"),
        ("$t$", r"$S_{\mathrm{cell}}$", "cell long axis vs. polarity\n(+ along, - across)"),
        ("$t$", r"$\langle a(\theta)\rangle_\ell$",
         "length-weighted alignment\n(parallel junctions being spent)")]):
    ax.set(xlabel=xlab, ylabel=ylab)
    ax.set_title(title, fontsize=10)
    ax.legend(frameon=False, fontsize=8)
    ax.spines["left"].set_visible(True); ax.spines["bottom"].set_visible(True)
fig.tight_layout()
fig.savefig(OUT_DIR / "cell_shape.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


### The law, read back off the run

The final check is that the tension the solver actually carried is the law from Step 2.
Recomputing $\theta$ from the recorded vertex positions and plotting it against the
recorded `line_tension` column has to land on the curve, at the start and at $t = t_f$ —
a mesh that has since been rewired by hundreds of T1s included. The points hug the curve
rather than sitting exactly on it, and that is the pipeline showing through: the tension a
frame carries was computed from the *previous* frame's geometry (the process reads the
`Tissue State` the solver published a step earlier), so each junction is scored at an angle
one $\mathrm{d}t$ stale. It also shows
where the junctions *are*: the histogram underneath is the length-weighted distribution of
$\theta$, and it moves towards $90°$ as the run proceeds.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.4, 6.6), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})
for col, key in enumerate(["px", "pxy"]):
    h, pol = histories[key], RUNS[key]["polarity"]
    p = np.asarray(pol, float); p = p / np.linalg.norm(p)
    edge, vert = h.datasets["edge"], h.datasets["vert"]
    times = sorted(vert["time"].unique())
    for t, colour, mark in [(first_active_time(h), "#0072B2", "."),
                            (times[-1], "#D55E00", ".")]:
        v = vert[vert["time"] == t].sort_values("vert")
        e = edge[edge["time"] == t]
        pos = np.empty((int(v["vert"].max()) + 1, 2))
        pos[v["vert"].to_numpy()] = v[COORDS].to_numpy(float)
        d = pos[e["trgt"].to_numpy()] - pos[e["srce"].to_numpy()]
        length = np.linalg.norm(d, axis=1)
        theta = np.degrees(np.arccos(np.clip(
            np.divide(np.abs(d @ p), length, out=np.zeros_like(length), where=length > 0),
            0, 1)))
        axes[0, col].plot(theta, e["line_tension"].to_numpy(), mark, ms=2.5,
                          alpha=0.35, color=colour, label=f"$t = {t:g}$")
        axes[1, col].hist(theta, bins=np.arange(0, 91, 3), weights=length,
                          histtype="step", lw=1.8, color=colour, density=True)
    th = np.linspace(0, 90, 361)
    axes[0, col].plot(th, tension_law(np.radians(th)), "k-", lw=1.4,
                      label=r"$\Lambda(\theta)$, Step 2")
    axes[0, col].set_title(RUNS[key]["label"], fontsize=10)
    axes[0, col].set(ylabel=r"recorded $\Lambda$", ylim=(-0.03, LAMBDA_MAX * 1.1))
    axes[0, col].legend(frameon=False, fontsize=8, markerscale=4)
    axes[1, col].set(xlabel=r"acute angle $\theta$ to $\hat p$ (deg)",
                     ylabel="junction length\n(normalised)",
                     xticks=[0, 15, 30, 45, 60, 75, 90])
for ax in axes.ravel():
    ax.spines["left"].set_visible(True); ax.spines["bottom"].set_visible(True)
fig.tight_layout()
fig.savefig(OUT_DIR / "tension_vs_angle.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


## Summary

A 2D vertex-model sheet with junction tension graded by the acute angle to a
user-supplied polarity vector undergoes convergent extension: it narrows along the
polarity axis and lengthens across it, and it does so about a **rotated** axis when the
polarity vector is rotated. An isotropic control with the same mean tension does neither,
and fires no T1s at all.

Two caveats the numbers above make plain, rather than results to quote without them:

* **how much of it is rearrangement depends on the polarity direction**, not just on the
  tension law. Along $(1,0)$ the deformation is essentially affine ($R \approx 0.04$, 22
  T1s) because the lattice's 90° junction family sits exactly at $\Lambda_{\min} = 0$ and is
  free to stretch. Along $(1,1)$ no family is free, and nearly two thirds of the shape
  change is carried by rearrangement ($R \approx 0.63$, 247 T1s) — intercalation, not
  stretching;
* the sheet still **shrinks** somewhat (total area $228 \to \sim 203$, against $215$ for the
  control): line tension has no opposing term at short junction lengths, so the sheet settles
  smaller than its rest area. The loss **saturates** — most of it is spent in the first
  $t \approx 40$ — no cell ends below 0.92 of the sheet's rest area, and the polygon
  distribution is essentially unchanged. The CE index is a ratio and unaffected; for `px` the
  split is $L_{\parallel}$ falling 34 % against $L_{\perp}$ growing 30 %.

The mechanism lives entirely in `DirectionalLineTension` — one process, driving the
existing `update_tension` and `apply_gradient` behaviors. `EulerSolver` is untouched.

Everything under `data/` and `outputs/` is git-ignored and regenerated by re-running this
notebook. To change the polarity, edit `POLARITIES` in Step 2; to change the law, edit
`LAMBDA_MIN` / `LAMBDA_MAX` / `PROFILE` / `SHARPNESS` in the same cell.
